"""
=============================================================================
MOVIES INDUSTRY & FINANCIAL ANALYTICS - END-TO-END ETL PIPELINE
=============================================================================
Author: Nihat
Description: This script processes 100 years of raw movie data (1923-2023).
             It performs string cleaning, currency conversion, JSON parsing, 
             data normalization (3NF), and exports 8 relational CSV tables 
             ready for PostgreSQL database import.
=============================================================================
"""

In [1]:
import pandas as pd
import numpy as np

# =============================================================================
# 1. DATA EXTRACTION & LOADING
# =============================================================================
# Load raw dataset and inspect initial structure

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Desktop\film\TMDB _IMDB_Movies_Dataset.csv")
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2858037,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2592086,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3214665,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1512631,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1576886,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [3]:
df.shape

(441084, 29)

In [4]:
df.info

<bound method DataFrame.info of             id                             title  vote_average  vote_count  \
0        27205                         Inception         8.364       34495   
1       157336                      Interstellar         8.417       32571   
2          155                   The Dark Knight         8.512       30619   
3        19995                            Avatar         7.573       29815   
4        24428                      The Avengers         7.710       29166   
...        ...                               ...           ...         ...   
441079  898453                Other Side of Love         0.000           0   
441080  898461                    Mind Mera Mind         0.000           0   
441081  898474     The Petersburg-Cannes Express         0.000           0   
441082  898396  A Very Short Film About Identity         0.000           0   
441083  898421                   Filling the Gap         0.000           0   

          status release_date  

In [5]:
df.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'tconst', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='str')

In [6]:
df.dtypes

id                        int64
title                       str
vote_average            float64
vote_count                int64
status                      str
release_date                str
revenue                   int64
runtime                   int64
adult                      bool
backdrop_path               str
budget                    int64
homepage                    str
tconst                      str
original_language           str
original_title              str
overview                    str
popularity              float64
poster_path                 str
tagline                     str
genres                      str
production_companies        str
production_countries        str
spoken_languages            str
keywords                    str
directors                   str
writers                     str
averageRating           float64
numVotes                  int64
cast                        str
dtype: object

# =============================================================================
# 2. DATA CLEANING & STANDARDIZATION
# =============================================================================
# Clean financial columns (budget, revenue) and handle null values
# ...

In [7]:
df.isnull().sum()

id                           0
title                        0
vote_average                 0
vote_count                   0
status                       0
release_date             24054
revenue                      0
runtime                      0
adult                        0
backdrop_path           254827
budget                       0
homepage                385736
tconst                       0
original_language            0
original_title               0
overview                 44354
popularity                   0
poster_path              78237
tagline                 347815
genres                   82495
production_companies    178377
production_countries    119775
spoken_languages        107934
keywords                269906
directors                 9965
writers                  64937
averageRating                0
numVotes                     0
cast                     69348
dtype: int64

In [8]:
cols_to_drop = ["homepage", "backdrop_path", "tagline", "keywords"]
df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

In [9]:
df.dropna(subset=["release_date"], inplace=True)

In [10]:
fill_rules = {
    "genres": "Unknown",
    "directors": "Unknown Director",
    "writers": "Unknown Writer",
    "cast": "Unknown Cast",
    "production_companies": "Independent / Unknown",
    "production_countries": "Unknown",
    "spoken_languages": "Unknown",
    "overview": "No overview available.",
    "poster_path": "No Poster",
}

df.fillna(value=fill_rules, inplace=True)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,poster_path,genres,production_companies,production_countries,spoken_languages,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,160000000,...,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili",Christopher Nolan,Christopher Nolan,8.8,2858037,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,165000000,...,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2592086,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,185000000,...,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3214665,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,237000000,...,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish",James Cameron,James Cameron,7.9,1512631,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,220000000,...,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1576886,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441079,898453,Other Side of Love,0.000,0,Released,2018-10-19,0,57,False,0,...,/sWn4XVBePSgJYY1E3a4m7d1r5Pi.jpg,Unknown,Independent / Unknown,Unknown,Unknown,Ambrish Bhatia,Ambrish Bhatia,4.2,13,Unknown Cast
441080,898461,Mind Mera Mind,0.000,0,Released,2021-06-06,0,0,False,0,...,/xKNLfPZUK6Q4YCpCksEqKy7w6xM.jpg,"Drama, Comedy",Independent / Unknown,Unknown,Unknown,"Harsh Agarwal, Pew Banerjee",Harsh Agarwal,6.4,9,Unknown Cast
441081,898474,The Petersburg-Cannes Express,0.000,0,Released,2003-09-19,0,100,False,0,...,No Poster,Unknown,Independent / Unknown,Unknown,Unknown,John Daly,"Hans Koningsberger, John Daly",3.6,29,Unknown Cast
441082,898396,A Very Short Film About Identity,0.000,0,Released,2012-06-22,0,9,False,0,...,/hz33d1iRd8oc4418wPI9QjNmFXy.jpg,Unknown,Independent / Unknown,Unknown,Unknown,Nick Anno,Nick Anno,8.8,14,Unknown Cast


In [11]:
df.isnull().sum()

id                      0
title                   0
vote_average            0
vote_count              0
status                  0
release_date            0
revenue                 0
runtime                 0
adult                   0
budget                  0
tconst                  0
original_language       0
original_title          0
overview                0
popularity              0
poster_path             0
genres                  0
production_companies    0
production_countries    0
spoken_languages        0
directors               0
writers                 0
averageRating           0
numVotes                0
cast                    0
dtype: int64

In [12]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year

In [13]:
df1 = df[(df["budget"] > 0) & (df["revenue"] > 0)]

In [14]:
df1["profit"] = df1["revenue"] - df1["budget"]

In [15]:
df1["roi_percentage"] = ((
    df1["profit"] / df1["budget"]
) * 100).round(2)

In [16]:
df1.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,production_countries,spoken_languages,directors,writers,averageRating,numVotes,cast,release_year,profit,roi_percentage
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,160000000,...,"United Kingdom, United States of America","English, French, Japanese, Swahili",Christopher Nolan,Christopher Nolan,8.8,2858037,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",2010,665532764,415.96
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,165000000,...,"United Kingdom, United States of America",English,Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2592086,"Matthew McConaughey, Anne Hathaway, Michael Ca...",2014,536729206,325.29
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,185000000,...,"United Kingdom, United States of America","English, Mandarin",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3214665,"Christian Bale, Heath Ledger, Aaron Eckhart, M...",2008,819558444,443.00
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,237000000,...,"United States of America, United Kingdom","English, Spanish",James Cameron,James Cameron,7.9,1512631,"Sam Worthington, Zoe Saldaña, Sigourney Weaver...",2009,2686706026,1133.63
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,220000000,...,United States of America,"English, Hindi, Russian",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1576886,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ...",2012,1298815515,590.37


In [17]:
df1.shape

(10213, 28)

In [18]:
df1["date_key"] = (
    df1["release_date"].dt.strftime("%Y%m%d").fillna("19000101").astype(int)
)

In [19]:
fact_movies = df1[[
    "id",
    "tconst",
    "title",
    "release_date",
    "date_key",
    "runtime",
    "budget",
    "revenue",
    "vote_average",
    "vote_count",
    "averageRating",
    "numVotes",
    "popularity",
    "original_language",
]].copy()

In [20]:
fact_movies.rename(columns={"id": "movie_id"}, inplace=True)

In [21]:
fact_movies.to_csv("fact_movies.csv", index=False)

In [22]:
unique_dates = df1["release_date"].dropna().drop_duplicates().sort_values()
dim_date = pd.DataFrame({"full_date": unique_dates})

In [23]:
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month  
dim_date["month_name"] = dim_date["full_date"].dt.strftime("%B")
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_of_week"] = dim_date["full_date"].dt.strftime("%A")

In [24]:
dim_date.to_csv("dim_date.csv", index=False)

# =============================================================================
# 3. JSON PARSING & ARRAY EXTRACTION
# =============================================================================
# Extract nested JSON structures for genres, directors, and main cast
# ...

In [27]:
def create_dim_and_bridge(df, source_col, entity_name, top_n=None):
  """Vergüllə ayrılmış mətni parçalayır, Unikal Dim və Bridge cədvəllərini qurur."""

  def split_items(val):
    if pd.isna(val) or str(val).strip() in [
        "",
        "Unknown",
        "Unknown Director",
        "Unknown Writer",
        "Unknown Cast",
    ]:
      return []
    items = [i.strip() for i in str(val).split(",")]
    return items[:top_n] if top_n else items

  temp_series = df[source_col].apply(split_items)
  exploded_df = df[["id"]].assign(entity=temp_series).explode("entity")
  exploded_df = exploded_df[
      exploded_df["entity"].notna() & (exploded_df["entity"] != "")
  ]

  # Unikal Dim Cədvəli
  unique_entities = pd.DataFrame(
      exploded_df["entity"].unique(), columns=[f"{entity_name}_name"]
  )
  unique_entities.sort_values(by=f"{entity_name}_name", inplace=True)
  unique_entities.reset_index(drop=True, inplace=True)
  unique_entities[f"{entity_name}_id"] = range(1, len(unique_entities) + 1)

  dim_table = unique_entities[[f"{entity_name}_id", f"{entity_name}_name"]]

  # Bridge Cədvəli
  bridge_table = pd.merge(
      exploded_df, dim_table, left_on="entity", right_on=f"{entity_name}_name"
  )
  bridge_table = bridge_table[["id", f"{entity_name}_id"]].rename(
      columns={"id": "movie_id"}
  )
  bridge_table.drop_duplicates(inplace=True)

  return dim_table, bridge_table

In [28]:
dim_genres, bridge_movie_genres = create_dim_and_bridge(df1, "genres", "genre")
dim_genres.to_csv("dim_genres.csv", index=False)
bridge_movie_genres.to_csv("bridge_movie_genres.csv", index=False)

In [29]:
dim_directors, bridge_movie_directors = create_dim_and_bridge(
    df1, "directors", "director"
)
dim_directors.to_csv("dim_directors.csv", index=False)
bridge_movie_directors.to_csv("bridge_movie_directors.csv", index=False)

In [30]:
dim_cast, bridge_movie_cast = create_dim_and_bridge(
    df1, "cast", "actor", top_n=5
)
dim_cast.to_csv("dim_cast.csv", index=False)
bridge_movie_cast.to_csv("bridge_movie_cast.csv", index=False)

# =============================================================================
# 4. CSV EXPORT (8 TABLES)
# =============================================================================
# Split into 3NF relational entity tables and save as processed CSVs
# ...

In [31]:
import pandas as pd
from sqlalchemy import create_engine


DB_USER = "postgres"
DB_PASSWORD = "admin123" 
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "movies_db"  


connection_url = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
engine = create_engine(connection_url)



load_sequence = [
    ("dim_date.csv", "dim_date"),
    ("dim_genres.csv", "dim_genres"),
    ("dim_directors.csv", "dim_directors"),
    ("dim_cast.csv", "dim_cast"),

    ("fact_movies.csv", "fact_movies"),

    ("bridge_movie_genres.csv", "bridge_movie_genres"),
    ("bridge_movie_directors.csv", "bridge_movie_directors"),
    ("bridge_movie_cast.csv", "bridge_movie_cast"),
]



print("🚀 CSV fayllarının PostgreSQL bazasına import prosesi başladı...\n")

for csv_file, table_name in load_sequence:
  try:
   
    df = pd.read_csv(csv_file)

   
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=2000, 
    )

    print(
        f"✅ {table_name:<23} <- {csv_file:<25} ({len(df):>6} sətir uğurla"
        " yazıldı)"
    )

  except Exception as e:
    print(
        f"\n❌ XƏTA: {table_name} cədvəlinə məlumat yazılarkən problem"
        " yarandı!"
    )
    print(f"Xəta təfərrüatı: {e}\n")
    break

print(
    "\n🎉 Bütün 8 cədvəlin məlumatları PostgreSQL bazasına tam və xətasız"
    " inteqrasiya olundu!"
)

🚀 CSV fayllarının PostgreSQL bazasına import prosesi başladı...

✅ dim_date                <- dim_date.csv              (  6246 sətir uğurla yazıldı)
✅ dim_genres              <- dim_genres.csv            (    19 sətir uğurla yazıldı)
✅ dim_directors           <- dim_directors.csv         (  5542 sətir uğurla yazıldı)
✅ dim_cast                <- dim_cast.csv              ( 20873 sətir uğurla yazıldı)

❌ XƏTA: fact_movies cədvəlinə məlumat yazılarkən problem yarandı!
Xəta təfərrüatı: Execution failed on sql 'INSERT INTO fact_movies (movie_id, tconst, title, release_date, date_key, runtime, budget, revenue, vote_average, vote_count, "averageRating", "numVotes", popularity, original_language) VALUES (:movie_id, :tconst, :title, :release_date, :date_key, :runtime, :budget, :revenue, :vote_average, :vote_count, :averageRating, :numVotes, :popularity, :original_language)': (psycopg2.errors.UndefinedColumn) column "averageRating" of relation "fact_movies" does not exist
LINE 1: ...ntime, bud

In [35]:
DB_USER = "postgres"
DB_PASSWORD = "admin123" 
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "movies_db"

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


with engine.connect() as conn:
  conn.execute(
      text(
          "TRUNCATE TABLE fact_movies, bridge_movie_genres,"
          " bridge_movie_directors, bridge_movie_cast CASCADE;"
      )
  )
  conn.commit()
print("🧹 Bazadakı yarımçıq cədvəllər tam təmizləndi!\n")


tables_to_load = [
    ("fact_movies.csv", "fact_movies", ["movie_id"]),
    ("bridge_movie_genres.csv", "bridge_movie_genres", ["movie_id", "genre_id"]),
    (
        "bridge_movie_directors.csv",
        "bridge_movie_directors",
        ["movie_id", "director_id"],
    ),
    ("bridge_movie_cast.csv", "bridge_movie_cast", ["movie_id", "actor_id"]),
]

print("🚀 Təmiz və təkrarsız yüklənmə başladı...\n")

for csv_file, table_name, pk_cols in tables_to_load:
  try:
    df = pd.read_csv(csv_file)

   
    df.drop_duplicates(subset=pk_cols, inplace=True)

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=2000,
    )
    print(
        f"✅ {table_name:<23} <- {csv_file:<25} ({len(df):>6} sətir uğurla"
        " yazıldı)"
    )
  except Exception as e:
    print(f"❌ Xəta baş verdi ({table_name}): {e}")
    break

print(
    "\n🎉 Bütün 8 cədvəlin daxilindəki məlumatlar PostgreSQL bazasına tam və"
    " xətasız inteqrasiya olundu!"
)

🧹 Bazadakı yarımçıq cədvəllər tam təmizləndi!

🚀 Təmiz və təkrarsız yüklənmə başladı...

✅ fact_movies             <- fact_movies.csv           ( 10122 sətir uğurla yazıldı)
✅ bridge_movie_genres     <- bridge_movie_genres.csv   ( 25057 sətir uğurla yazıldı)
✅ bridge_movie_directors  <- bridge_movie_directors.csv ( 11281 sətir uğurla yazıldı)
✅ bridge_movie_cast       <- bridge_movie_cast.csv     ( 49564 sətir uğurla yazıldı)

🎉 Bütün 8 cədvəlin daxilindəki məlumatlar PostgreSQL bazasına tam və xətasız inteqrasiya olundu!
